In [ ]:
import base64
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")

username = "AnikMallick"
repo_name = "bn-proverbs-in-context"

repo_dir = Path("/kaggle/working") / repo_name

auth = base64.b64encode(
    f"{username}:{token}".encode()
).decode()

try:
    subprocess.run(
        [
            "git",
            "-c",
            f"http.https://github.com/.extraheader="
            f"AUTHORIZATION: basic {auth}",
            "clone",
            f"https://github.com/{username}/{repo_name}.git",
            str(repo_dir),
        ],
        check=True,
    )
finally:
    del token
    del auth

print("Repository cloned successfully.")

In [ ]:
%cd /kaggle/working/bn-proverbs-in-context

!pip install -e .

In [1]:
from pathlib import Path
from probad.utils import print_div
from probad.schema import TestItem, validate_items
import json
from typing import Any
import probad.builder as exp
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
DATA_PATH = Path("/kaggle/input/datasets/amallick0507/probad")

train_data = list(TestItem.load_items(DATA_PATH/"train.jsonl"))
test_data = list(TestItem.load_items(DATA_PATH/"test.jsonl"))
val_data = list(TestItem.load_items(DATA_PATH/"val.jsonl"))


source = DATA_PATH / "stopwords-bn.json"

try:
    data: Any = json.loads(source.read_text(encoding="utf-8-sig"))
except FileNotFoundError as exc:
    raise FileNotFoundError(f"Stopword file not found: {source}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Invalid JSON in stopword file {source}: "
        f"line {exc.lineno}, column {exc.colno}: {exc.msg}"
    ) from exc

BN_STOPWORDS =  frozenset([word.strip() for word in data])

print_div("train")
for issue in validate_items(train_data, BN_STOPWORDS):
    print(issue)

print_div("test")
for issue in validate_items(test_data, BN_STOPWORDS):
    print(issue)

print_div("val")
for issue in validate_items(val_data, BN_STOPWORDS):
    print(issue)

=============================================> train <==============================================
==============================================> test <==============================================
[WARNING] prv_088_1: en.correct_echoes_gloss: correct option reuses many literal-gloss words
[WARNING] prv_088_2: en.correct_echoes_gloss: correct option reuses many literal-gloss words
==============================================> val <===============================================


In [2]:
import os
import torch
import json
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_DIR = "/kaggle/working/qwen3b-lora-fixed"

MAX_SEQ_LENGTH = 512

print(torch.cuda.get_device_name(0)), print(torch.cuda.device_count())  

Tesla T4
1


(None, None)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map={"": 0},
#     torch_dtype=torch.float16,
#     trust_remote_code=True
# )

# model.config.use_cache = False

# model = prepare_model_for_kbit_training(
#     model,
#     use_gradient_checkpointing=True
# )

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16,
                                bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config,
                                             device_map={"": 0}, dtype=torch.float16)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

In [8]:
import json
from collections import Counter
import pandas as pd
from datasets import Dataset
import probad.builder as exp
from functools import partial

TASKS = {"x1": exp.x1_correct_vs_literal, "x5": exp.x5_three_way, "x2": exp.x2_negative}   # x3, x4, x7 stay untrained (transfer tests)


def build(fn, item, lang, seed):
    """-> (prompt, gold letter in upper case) or None. Accepts the builder's 2-, 3- or 4-tuple."""
    out = fn(item, lang=lang, seed=seed)
    if out is None:
        return None
    out = tuple(out)
    if len(out) == 3 and not isinstance(out[2], (list, tuple)):   # (system, prompt, gold)
        out = out[1:]
    return out[0], str(out[1]).strip().upper()


def order_seeds(fn, item, lang, max_seed=300):
    """Seed 0 plus the first seeds that move the gold letter to every other position (same as scoring)."""
    first = build(fn, item, lang, 0)
    if first is None:
        return []
    n = sum(1 for L in "ABC" if f"\n{L}. " in "\n" + first[0])
    seeds, golds = [0], {first[1]}
    for s in range(1, max_seed):
        b = build(fn, item, lang, s)
        if b and b[1] not in golds:
            seeds.append(s); golds.add(b[1])
        if len(golds) >= n:
            break
    return seeds


def to_text(tok, prompt, system=None):
    """Exactly what the run notebook scores for a chat model: chat template, then the prompt's answer cue
    ("Answer:" / "উত্তর:") at the start of the reply. The next token is the letter."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return text + prompt.rstrip().rsplit("\n", 1)[-1]


def make_sft_dataset(items, tok, tasks=("x1", "x5", "x2"), langs=("en", "bn"), swap=True, out=None):
    """items: one split (dicts or TestItems). Returns a datasets.Dataset with prompt / completion (" A")
    for TRL's SFTTrainer, plus columns for checking (item_id, task, lang, order, gold)."""
    rows = []
    for it in items:
        it = it.model_dump() if hasattr(it, "model_dump") else it
        for task in tasks:
            for lang in langs:
                seeds = order_seeds(TASKS[task], it, lang) if swap else [0]
                for order, s in enumerate(seeds):
                    prompt, gold = build(TASKS[task], it, lang, s)
                    rows.append({"prompt": to_text(tok, prompt), "completion": " " + gold,
                                 "item_id": it["item_id"], "proverb_id": it["proverb"]["proverb_id"],
                                 "task": task, "lang": lang, "seed": s, "order": order, "gold": gold})
    df = pd.DataFrame(rows)

    n_tok = df["prompt"].map(lambda t: len(tok(t, add_special_tokens=False).input_ids))
    print(f"{len(df)} examples from {df.item_id.nunique()} items / {df.proverb_id.nunique()} proverbs | "
          f"tokens mean {n_tok.mean():.0f}, max {n_tok.max()}")
    print(df.groupby(["task", "lang"]).size().rename("examples").to_dict())
    print("gold letters per task (should be balanced):", {t: dict(Counter(g)) for t, g in df.groupby("task")["gold"]})
    if out:
        df.to_json(out, orient="records", lines=True, force_ascii=False)
        print("saved", out)
    return Dataset.from_pandas(df, preserve_index=False)

In [9]:
pid = lambda xs: {(x.model_dump() if hasattr(x, "model_dump") else x)["proverb"]["proverb_id"] for x in xs}
assert not (pid(train_data) | pid(val_data)) & pid(test_data), "train/val proverbs appear in test"

train_ds = make_sft_dataset(train_data, tokenizer, out="/kaggle/working/sft_train.jsonl")
val_ds = make_sft_dataset(val_data, tokenizer, out="/kaggle/working/sft_val.jsonl")

# SFTTrainer only needs these two columns
train_ds, val_ds = train_ds.select_columns(["prompt", "completion"]), val_ds.select_columns(["prompt", "completion"])

4200 examples from 300 items / 100 proverbs | tokens mean 256, max 511
{('x1', 'bn'): 600, ('x1', 'en'): 600, ('x2', 'bn'): 600, ('x2', 'en'): 600, ('x5', 'bn'): 900, ('x5', 'en'): 900}
gold letters per task (should be balanced): {'x1': {'B': 600, 'A': 600}, 'x2': {'A': 600, 'B': 600}, 'x5': {'C': 600, 'B': 600, 'A': 600}}
saved /kaggle/working/sft_train.jsonl
1050 examples from 75 items / 25 proverbs | tokens mean 260, max 513
{('x1', 'bn'): 150, ('x1', 'en'): 150, ('x2', 'bn'): 150, ('x2', 'en'): 150, ('x5', 'bn'): 225, ('x5', 'en'): 225}
gold letters per task (should be balanced): {'x1': {'A': 150, 'B': 150}, 'x2': {'B': 150, 'A': 150}, 'x5': {'A': 150, 'B': 150, 'C': 150}}
saved /kaggle/working/sft_val.jsonl


In [10]:
train_ds[0], val_ds[0]

({'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nRajesh: Sumon trolled his ex-colleague online for weeks, and now she has posted all his old embarrassing messages.\nIpsita: Itti marle patkelti khete hoy.\n\nQuestion: What does Ipsita mean?\nA. Throw a brick at someone and a stone comes back at you.\nB. Sumon hurt her first, so he should have expected to be hurt back.\nAnswer:<|im_end|>\n<|im_start|>assistant\nAnswer:',
  'completion': ' B'},
 {'prompt': "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nAnkush: Montu went to rehab for six months and came back promising to change, and last night he was caught stealing from his own mother again.\nTuli: Koyla dhule moyla jay na.\n\nQuestion: What does Tuli mean?\nA. Montu's basic nature hasn't changed, whatever he promises.\nB. However much you wash a piece of coal, its blackness will never 

In [11]:
from trl import SFTConfig, SFTTrainer
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,          # effective batch 16 -> ~190 steps per epoch
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    fp16=True, bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},   # avoids the "no grad" problem with PEFT

    # data: prompt/completion columns, loss on the answer letter only
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    completion_only_loss=True,

    # keep the best checkpoint, not the last one
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,

    logging_steps=10,
    report_to="none",
    dataloader_num_workers=2,
    seed=0,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

Adding EOS to train dataset:   0%|          | 0/4200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4200 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4200 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1050 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1050 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1050 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1050 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1050 [00:00<?, ? examples/s]

In [12]:
b = trainer.data_collator([trainer.train_dataset[0]])
print(repr(tokenizer.decode(b["input_ids"][0][b["labels"][0] != -100])))



' B<|im_end|>'


In [14]:
for n, p in trainer.model.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()
print({str(p.dtype) for p in trainer.model.parameters() if p.requires_grad})

{'torch.float32'}


In [15]:
trainer.train()
trainer.save_model(OUTPUT_DIR + "/best")

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.046306,0.113819,0.006503,0.983779,212502.000000
100,0.000027,0.035820,0.000875,0.995706,420328.000000
150,0.000791,0.025953,0.015735,0.989981,627945.000000
200,0.000490,0.033098,0.001518,0.995229,835303.000000
250,0.000024,0.039687,0.001168,0.995229,1038635.000000
300,0.000019,0.061357,0.001624,0.993798,1239132.000000
350,0.000016,0.060174,0.001659,0.993798,1447530.000000
400,0.000023,0.052531,0.001155,0.995229,1650291.000000
450,0.000013,0.048171,0.001270,0.995706,1857769.000000
500,0.000012,0.048244,0.001230,0.995706,2062790.000000


In [ ]:
import os
import tempfile
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN_WRITE")
api = HfApi(token=token)

REPO_ID = "anik-owl/qwen2.5-3b-instruct-bn-proverbs-lora"

files_to_move = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "chat_template.jinja",
    "tokenizer.json",
    "tokenizer_config.json",
    "train_curve.png",
    "train_info.json",
    "train_log.csv",
    "training_args.bin",
]


repo_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="model"
)

missing = [f for f in files_to_move if f not in repo_files]

if missing:
    raise RuntimeError(f"Files missing from repository root: {missing}")

# Avoid overwriting an existing v1
existing_v1 = [f"v1/{f}" for f in files_to_move if f"v1/{f}" in repo_files]

if existing_v1:
    raise RuntimeError(f"Some v1 files already exist. Aborting: {existing_v1}")

print("All source files verified.")

with tempfile.TemporaryDirectory() as tmpdir:

    print("Downloading original files...")

    for filename in files_to_move:
        downloaded_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=filename,
            repo_type="model",
            token=token,
        )

        # Copy to a flat temporary directory
        import shutil
        shutil.copy2(downloaded_path, os.path.join(tmpdir, filename))

    print("All files downloaded.")

    api.upload_folder(
        folder_path=tmpdir,
        repo_id=REPO_ID,
        repo_type="model",
        path_in_repo="v1",
        commit_message="Create v1: archive original LoRA experiment - (x1+x5, train split)",
    )

    print("Uploaded files to v1/.")

repo_files_after_upload = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="model"
)

verified = [f"v1/{f}" for f in files_to_move]

missing_v1 = [f for f in verified if f not in repo_files_after_upload]

if missing_v1:
    raise RuntimeError(
        f"Verification failed. Files missing in v1: {missing_v1}\n"
        "Root files have NOT been deleted."
    )

print("All v1 files verified successfully.")


In [22]:
files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="model"
)

for f in files:
    print(f)

.gitattributes
README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
tokenizer.json
tokenizer_config.json
train_curve.png
train_info.json
train_log.csv
training_args.bin
v1/adapter_config.json
v1/adapter_model.safetensors
v1/chat_template.jinja
v1/tokenizer.json
v1/tokenizer_config.json
v1/train_curve.png
v1/train_info.json
v1/train_log.csv
v1/training_args.bin


In [34]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# login(token=UserSecretsClient().get_secret("HF_TOKEN_WRITE"))
# api = HfApi()

REPO_ID = "anik-owl/qwen2.5-3b-instruct-bn-proverbs-lora"
ADAPTER_DIR = "/kaggle/working/qwen3b-lora-fixed/best"

# api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(
    folder_path=ADAPTER_DIR, repo_id = REPO_ID,
    path_in_repo ="v2", commit_message="QLoRA adapter (x1 + x2 + x5, train split)",repo_type="model",
    ignore_patterns=["checkpoint-*", "optimizer.pt", "scheduler.pt", "rng_state*", "*.bin.tmp"],
)
print("https://huggingface.co/" + REPO_ID)

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6b016ffa-748967a75f6ee2fa2bc17ab9;774895a6-fa7d-421a-b5a5-22093918d322)

Repository Not Found for url: https://huggingface.co/api/models/anik-owl/qwen2.5-3b-instruct-bn-proverbs-lora/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [26]:
import json, platform, datetime
from pathlib import Path
import pandas as pd, torch, transformers, peft, trl
OUT = Path("/kaggle/working/")

def data_mix(ds):
    """Counts per task / language / gold letter if those columns are still in the dataset."""
    cols = set(ds.column_names)
    if not {"task", "lang"} <= cols:
        return {"examples": len(ds)}
    df = ds.to_pandas()
    return {"examples": len(df), "items": int(df["item_id"].nunique()) if "item_id" in cols else None,
            "proverbs": int(df["proverb_id"].nunique()) if "proverb_id" in cols else None,
            "per_task_lang": {f"{t}_{l}": int(n) for (t, l), n in df.groupby(["task", "lang"]).size().items()},
            "gold_letters": df["gold"].value_counts().to_dict() if "gold" in cols else None}

log = pd.DataFrame(trainer.state.log_history)
log.to_csv(OUT / "train_log.csv", index=False)
final = log.dropna(subset=["train_runtime"]).iloc[-1].to_dict() if "train_runtime" in log else {}
ev = log.dropna(subset=["eval_loss"]) if "eval_loss" in log else pd.DataFrame()

model_cfg = trainer.model.config
qcfg = getattr(model_cfg, "quantization_config", None)
info = {
    "saved_at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "base_model": "Qwen/Qwen2.5-3B-Instruct",
    "quantization": qcfg.to_dict() if hasattr(qcfg, "to_dict") else qcfg,
    "lora": {k: (sorted(v) if isinstance(v, set) else v)
             for k, v in trainer.model.peft_config["default"].to_dict().items()},
    "trainable_params": sum(p.numel() for p in trainer.model.parameters() if p.requires_grad),
    "total_params": sum(p.numel() for p in trainer.model.parameters()),
    "training_args": {k: v for k, v in trainer.args.to_dict().items()
                      if not k.startswith(("hub_", "push_to_hub", "logging_dir"))},
    "data": {
        "description": "x1 (correct vs literal) + x5 (3 options), en + bn conversations, all option orders "
                       "(gold in every position); prompt = scoring text ending in the answer cue, "
                       "completion = answer letter; loss on the completion only",
        "train": data_mix(train_ds_full if "train_ds_full" in globals() else trainer.train_dataset),
        "val": data_mix(val_ds_full if "val_ds_full" in globals() else trainer.eval_dataset),
        "test_proverbs_overlap": 0,        # checked with the assert before training
    },
    "result": {
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_eval_loss": trainer.state.best_metric,
        "first_eval_loss": float(ev["eval_loss"].iloc[0]) if len(ev) else None,
        "last_eval_loss": float(ev["eval_loss"].iloc[-1]) if len(ev) else None,
        "global_steps": trainer.state.global_step, "epochs_run": trainer.state.epoch,
        "train_runtime_min": round(final.get("train_runtime", 0) / 60, 1),
        "final_train_loss": final.get("train_loss"),
    },
    "environment": {
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "peft": peft.__version__, "trl": trl.__version__,
        "bitsandbytes": __import__("bitsandbytes").__version__,
    },
}
json.dump(info, open(OUT / "train_info.json", "w"), indent=2, default=str)
print(json.dumps({k: info[k] for k in ("base_model", "result", "data")}, indent=2, default=str))

{
  "base_model": "Qwen/Qwen2.5-3B-Instruct",
  "result": {
    "best_checkpoint": "/kaggle/working/qwen3b-lora-fixed/checkpoint-150",
    "best_eval_loss": 0.025952918455004692,
    "first_eval_loss": 0.11381945759057999,
    "last_eval_loss": 0.04797668755054474,
    "global_steps": 789,
    "epochs_run": 3.0,
    "train_runtime_min": 179.5,
    "final_train_loss": 0.14588873671780972
  },
  "data": {
    "description": "x1 (correct vs literal) + x5 (3 options), en + bn conversations, all option orders (gold in every position); prompt = scoring text ending in the answer cue, completion = answer letter; loss on the completion only",
    "train": {
      "examples": 4200
    },
    "val": {
      "examples": 1047
    },
    "test_proverbs_overlap": 0
  }
}


In [28]:
api = HfApi()
for f in ["train_info.json", "train_log.csv", "train_curve.png"]:
    if (OUT / f).exists():
        api.upload_file(path_or_fileobj=str(OUT / f), path_in_repo=REPO_ID + "/v2", repo_id=REPO_ID)
        print("uploaded", f)

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6b016f03-10b789d1375fc5be2c416acb;ccc61902-ced5-4c3b-9d8d-6593cdde7bda)

Repository Not Found for url: https://huggingface.co/api/models/anik-owl/qwen2.5-3b-instruct-bn-proverbs-lora/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.